# Mission 3 · 증상 9클래스 multi-label (Colab GPU)로컬 CPU 베이스라인(TF-IDF + LogReg)은 macro-F1 **0.5848**. 이 노트북은 그 위에KLUE-RoBERTa / Kc-ELECTRA 파인튜닝을 올린다.**데이터 경로**: 로컬에서 `python -m src.preprocess.pack_for_colab --what m3` 로 만든`m3_text.json.gz` (14 MB) 를 Google Drive 의 `MyDrive/dcc/` 에 올려둘 것.전사 텍스트에는 개인정보가 포함되므로 **GitHub 에는 절대 올리지 않는다** — Drive 경유만 사용.

In [ ]:
import torch, subprocessprint(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],                     capture_output=True, text=True).stdout)print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
from google.colab import drivedrive.mount('/content/drive')DATA = '/content/drive/MyDrive/dcc/m3_text.json.gz'CKPT = '/content/drive/MyDrive/dcc/ckpt'import os; os.makedirs(CKPT, exist_ok=True)

In [ ]:
!pip -q install "transformers>=4.44" "accelerate>=0.33" scikit-learn

## 저장소 코드 불러오기지표 계산과 그림 그리기는 **로컬 베이스라인과 똑같은 코드**를 써야 한다.노트북에 따로 구현하면 GPU 결과와 CPU 기준선이 다른 계산으로 나와 비교가 무의미해진다.`src.zip`(49 KB)을 Drive 의 `MyDrive/dcc/` 에 함께 올려 두고 여기서 풀어 쓴다.(GitHub 에 push 했다면 `!git clone` 으로 대체해도 된다.)

In [ ]:
import sys, zipfile, osos.makedirs('/content/repo', exist_ok=True)with zipfile.ZipFile(f'{DIR}/src.zip') as z:    z.extractall('/content/repo')sys.path.insert(0, '/content/repo')# 로컬과 동일한 지표 구현을 그대로 가져온다 (출제 PDF 10쪽 절차)from src.common.metrics import macro_f1_pdf, tune_thresholdsfrom src.common.paths import SYMPTOM_9 as SYM_FROM_SRCprint('불러온 클래스:', SYM_FROM_SRC)

In [ ]:
import gzip, json, numpy as npwith gzip.open(DATA, 'rt', encoding='utf-8') as f:    D = json.load(f)SYMPTOM_9 = ["고열","구토","두통","복통","어지러움","열상","오심","전신쇠약","호흡곤란"]tr, va = D['train'], D['val']Xtr = [r['text'] for r in tr];  Ytr = np.array([r['y'] for r in tr], dtype=np.float32)Xva = [r['text'] for r in va];  Yva = np.array([r['y'] for r in va], dtype=np.float32)# Training 안에서 dev 분리 — 임계값 튜닝 전용. Validation 은 학습/튜닝에 절대 사용 금지.rng = np.random.default_rng(0); perm = rng.permutation(len(tr)); n_dev = len(tr)//10dev_i, fit_i = perm[:n_dev], perm[n_dev:]print(f'fit={len(fit_i)}  dev={len(dev_i)}  val={len(Xva)}  labels={Ytr.shape}')print('클래스별 양성 비율:', dict(zip(SYMPTOM_9, (Ytr.mean(0)*100).round(1))))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification# klue/roberta-base: KLUE 벤치마크 기준 한국어 이해 태스크 전반에서 안정적.# 대안 'beomi/KcELECTRA-base-v2022' 는 구어체 말뭉치로 학습돼 있어 통화 전사문과 결이 맞고,# AI-Hub 가 이 데이터셋의 공식 베이스라인으로 쓴 Kc-ELECTRA 계열이다. 둘 다 돌려 비교할 것.MODEL = 'klue/roberta-base'# 전체 대화 길이는 중앙값 397자 / p95 842자 / 최대 1,292자 (로컬 실측).# 512 토큰이면 대부분 들어가지만 상위 5% 가량은 뒤가 잘린다.# 증상은 통화 초반(신고 사유 진술)에 몰리므로 앞을 남기는 기본 truncation 이 유리하다.MAXLEN = 512tok = AutoTokenizer.from_pretrained(MODEL)# problem_type='multi_label_classification' 이 핵심.# 이 값을 주면 HF 가 손실을 BCEWithLogitsLoss 로 바꾸고 9개 라벨을 각각 독립 판정한다.# (기본값이면 softmax CrossEntropy 가 걸려 "9개 중 정확히 1개" 문제로 잘못 풀린다)model = AutoModelForSequenceClassification.from_pretrained(    MODEL, num_labels=9, problem_type='multi_label_classification').cuda()print(f'{MODEL}  파라미터 {sum(p.numel() for p in model.parameters())/1e6:.0f}M')

In [ ]:
import torchfrom torch.utils.data import Dataset, DataLoaderclass DS(Dataset):    """토큰화는 collate 에서 배치 단위로 한다 — 배치 안의 최대 길이에만 padding 해 낭비를 줄인다."""    def __init__(self, texts, y):        self.t, self.y = texts, y    def __len__(self):        return len(self.t)    def __getitem__(self, i):        return self.t[i], self.y[i]def collate(batch):    txt = [b[0] for b in batch]    # 라벨은 float 여야 한다. multi_label 모드는 BCEWithLogitsLoss 를 쓰는데 이 손실은    # 정수 타깃을 받지 않는다 (단일 클래스 CrossEntropy 와 다른 점).    y = torch.tensor(np.stack([b[1] for b in batch]))    enc = tok(txt, truncation=True, max_length=MAXLEN, padding=True, return_tensors='pt')    enc['labels'] = y    return enc# fit = 실제 가중치 학습, dev = 임계값 튜닝, val = 최종 보고. 세 집합의 역할을 섞지 않는다.fit_dl = DataLoader(DS([Xtr[i] for i in fit_i], Ytr[fit_i]), batch_size=16,                    shuffle=True, collate_fn=collate, num_workers=2)dev_dl = DataLoader(DS([Xtr[i] for i in dev_i], Ytr[dev_i]), batch_size=32,                    collate_fn=collate, num_workers=2)val_dl = DataLoader(DS(Xva, Yva), batch_size=32, collate_fn=collate, num_workers=2)# 학습 곡선용: 에폭마다 train 쪽 손실/F1 을 '평가 모드'로 다시 재기 위한 로더.# fit 전체(26k)를 매 에폭 평가하면 비싸므로 val 과 비슷한 크기로 무작위 추출해 쓴다.# (곡선의 목적은 절대값이 아니라 train-val 격차의 추세를 보는 것이다)fit_eval_i = fit_i[:len(Xva)]fit_eval_dl = DataLoader(DS([Xtr[i] for i in fit_eval_i], Ytr[fit_eval_i]), batch_size=32,                         collate_fn=collate, num_workers=2)

In [ ]:
from torch.optim import AdamWfrom transformers import get_linear_schedule_with_warmupEPOCHS = 3      # 26k 샘플 기준 3 epoch 이면 수렴한다. 더 돌리면 희소 클래스에 과적합되기 쉽다.# lr 2e-5 는 BERT 계열 파인튜닝의 표준값. weight_decay 는 LayerNorm/bias 에도 걸리지만# 이 규모에서는 영향이 미미해 단순하게 둔다.opt = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)# 선형 감쇠 + 초반 6% warmup. warmup 없이 시작하면 초기 큰 gradient 가 사전학습 표현을 망친다.steps = len(fit_dl) * EPOCHSsched = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)scaler = torch.amp.GradScaler('cuda')      # 혼합정밀: T4 에서 메모리 절감 + 속도 향상@torch.no_grad()def predict(dl):    """(n, 9) 확률 행렬을 반환. 임계값은 적용하지 않는다 — 튜닝은 나중 셀에서 한다."""    model.eval()    out = []    for b in dl:        b = {k: v.cuda() for k, v in b.items()}        with torch.amp.autocast('cuda'):            # labels 를 빼고 넘겨 손실 계산을 건너뛴다.            logits = model(**{k: v for k, v in b.items() if k != 'labels'}).logits        # multi-label 이므로 softmax 가 아니라 sigmoid. 각 라벨이 독립 확률이다.        out.append(torch.sigmoid(logits.float()).cpu().numpy())    return np.concatenate(out)@torch.no_grad()def eval_loss_and_f1(dl, y_true):    """에폭 끝마다 호출: 해당 split 의 평균 손실과 macro-F1(임계값 0.5)을 함께 잰다."""    from sklearn.metrics import f1_score    model.eval()    tot, n, probs = 0.0, 0, []    for b in dl:        b = {k: v.cuda() for k, v in b.items()}        with torch.amp.autocast('cuda'):            out = model(**b)        tot += out.loss.item() * b['labels'].size(0)        n += b['labels'].size(0)        probs.append(torch.sigmoid(out.logits.float()).cpu().numpy())    p = (np.concatenate(probs) >= 0.5).astype(int)    f1 = float(np.mean([f1_score(y_true[:, c], p[:, c], zero_division=0) for c in range(9)]))    return tot / n, f1# 에폭별 곡선을 남긴다. train/val 손실이 갈라지기 시작하는 지점이 과적합 시작점이고,# 거기서 EPOCHS 를 줄이거나 early stopping 을 걸면 된다.hist = {'epoch': [], 'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}for ep in range(EPOCHS):    model.train()    tot = 0.0    for i, b in enumerate(fit_dl):        b = {k: v.cuda() for k, v in b.items()}        opt.zero_grad(set_to_none=True)        with torch.amp.autocast('cuda'):            loss = model(**b).loss      # labels 가 있으면 BCEWithLogitsLoss 를 내부 계산        scaler.scale(loss).backward()        scaler.step(opt)        scaler.update()        sched.step()        tot += loss.item()        if (i+1) % 200 == 0:            print(f'ep{ep+1} step {i+1}/{len(fit_dl)} loss {tot/(i+1):.4f}', flush=True)    # train 손실은 학습 중 누적값(드롭아웃 켜진 상태)이라 val 과 직접 비교가 어렵다.    # 공정한 비교를 위해 eval 모드로 fit split 을 다시 한 번 잰다.    trl, trf = eval_loss_and_f1(fit_eval_dl, Ytr[fit_eval_i])    val, vaf = eval_loss_and_f1(val_dl, Yva)    hist['epoch'].append(ep + 1)    hist['train_loss'].append(trl); hist['val_loss'].append(val)    hist['train_f1'].append(trf);   hist['val_f1'].append(vaf)    print(f'== epoch {ep+1}  train loss {trl:.4f} / F1 {trf:.4f}   '          f'val loss {val:.4f} / F1 {vaf:.4f}')

In [ ]:
# ---- 학습 곡선: train vs validation ----import matplotlib.pyplot as plt, matplotlib!apt-get -qq install fonts-nanum > /dev/nullmatplotlib.font_manager.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')matplotlib.rc('font', family='NanumGothic'); matplotlib.rc('axes', unicode_minus=False)fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))e = hist['epoch']# 왼쪽: 손실. 두 곡선이 벌어지기 시작하면 과적합이 시작된 것이다.a1.plot(e, hist['train_loss'], 'o-', color='#9FB6C9', label='Training')a1.plot(e, hist['val_loss'], 'o-', color='#1B5E96', label='Validation')a1.set_xlabel('에폭'); a1.set_ylabel('BCE 손실'); a1.set_title('손실')a1.legend(frameon=False); a1.grid(axis='y', alpha=.3)# 오른쪽: macro-F1. 실제 채점 지표이므로 이쪽이 모델 선택의 기준이다.a2.plot(e, hist['train_f1'], 'o-', color='#9FB6C9', label='Training')a2.plot(e, hist['val_f1'], 'o-', color='#1B5E96', label='Validation')a2.axhline(0.5848, color='#2A6B48', ls=':', label='로컬 CPU 베이스라인')a2.set_xlabel('에폭'); a2.set_ylabel('macro-F1'); a2.set_title('macro-F1 (임계값 0.5)')a2.legend(frameon=False); a2.grid(axis='y', alpha=.3)fig.tight_layout(); plt.show()best = int(np.argmax(hist['val_f1'])) + 1print(f"val macro-F1 최고: epoch {best} ({max(hist['val_f1']):.4f})")if best < len(e):    print(f"→ 이후 에폭에서 하락했다면 EPOCHS 를 {best} 로 줄이는 편이 낫다.")

In [ ]:
from sklearn.metrics import f1_score# macro_f1_pdf 와 tune_thresholds 는 위에서 src.zip 에서 불러왔다.# 노트북에 다시 구현하지 않는 이유: 로컬 CPU 베이스라인(0.5848)과 반드시 같은 계산이어야# 두 숫자를 비교할 수 있다. 구현이 갈리면 비교 자체가 무의미해진다.def macro_f1(y_true, y_pred):    return macro_f1_pdf(y_true.astype(int), y_pred.astype(int))[0]dev_s, val_s = predict(dev_dl), predict(val_dl)# 클래스별 임계값을 dev 에서만 튜닝한다. Validation 으로 튜닝하면 점수가 낙관적으로 부풀려진다.th = tune_thresholds(Ytr[dev_i].astype(int), dev_s)pred05 = (val_s >= 0.5).astype(int)predth = (val_s >= th[None,:]).astype(int)print(f'Validation macro-F1  @0.5={macro_f1(Yva, pred05):.4f}   @tuned={macro_f1(Yva, predth):.4f}')print('로컬 CPU 베이스라인(TF-IDF+LogReg) = 0.5848  <- 이 값을 넘어야 GPU 도입이 정당화된다')# 로컬 베이스라인의 클래스별 F1. 어느 클래스가 개선됐는지 직접 비교하기 위해 박아둔다.LOCAL = {'고열':0.657,'구토':0.487,'두통':0.438,'복통':0.768,'어지러움':0.617,         '열상':0.846,'오심':0.300,'전신쇠약':0.541,'호흡곤란':0.610}for c, s, t in zip(SYMPTOM_9, [f1_score(Yva[:,i], predth[:,i], zero_division=0) for i in range(9)], th):    print(f'  {c:6s} F1={s:.3f}  th={t:.2f}   (로컬 {LOCAL[c]:.3f}, 차이 {s-LOCAL[c]:+.3f})')

In [ ]:
import matplotlib.pyplot as pltimport matplotlib!apt-get -qq install fonts-nanum > /dev/nullmatplotlib.font_manager.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')matplotlib.rc('font', family='NanumGothic'); matplotlib.rc('axes', unicode_minus=False)fig, axes = plt.subplots(3, 3, figsize=(10.5, 10))for k, ax in enumerate(axes.ravel()):    t, p = Yva[:,k].astype(int), predth[:,k]    cm = np.array([[((t==0)&(p==0)).sum(), ((t==0)&(p==1)).sum()],                   [((t==1)&(p==0)).sum(), ((t==1)&(p==1)).sum()]], dtype=float)    pct = cm / np.maximum(cm.sum(1, keepdims=True), 1)    ax.imshow(pct, cmap='Blues', vmin=0, vmax=1)    ax.set_xticks([0,1], ['없음','있음']); ax.set_yticks([0,1], ['없음','있음'])    ax.set_title(f"{SYMPTOM_9[k]}  F1={f1_score(t, p, zero_division=0):.2f}", fontsize=10)    for i in range(2):        for j in range(2):            ax.text(j, i, f'{int(cm[i,j]):,}\n{pct[i,j]*100:.1f}%', ha='center', va='center',                    fontsize=9, color='white' if pct[i,j] > 0.55 else '#16202B')fig.suptitle(f'Mission 3 · KLUE-RoBERTa 혼동행렬 (macro-F1={macro_f1(Yva, predth):.4f})', fontsize=13)fig.tight_layout(); plt.show()

In [ ]:
torch.save({'state_dict': model.state_dict(), 'thresholds': th,            'model_name': MODEL, 'classes': SYMPTOM_9, 'maxlen': MAXLEN},           f'{CKPT}/m3.pt')print('저장 완료:', f'{CKPT}/m3.pt')

## 예측을 로컬과 같은 형식으로 저장`cache/m3_val_pred.npz` 의 스키마(y_true / y_score / y_pred / thresholds / classes)를 그대로 맞춘다.이 파일을 로컬로 내려받아 `cache/` 에 넣으면, 로컬에서 이미 쓰던`python -m src.viz.plots m3` 와 `python -m src.viz.inspect_errors m3 --cls 오심` 이**GPU 결과에 대해서도 그대로 동작한다.** 혼동행렬을 같은 코드로 그려야 CPU 기준선과 비교된다.

In [ ]:
np.savez_compressed(f'{DIR}/m3_val_pred_gpu.npz',                    y_true=Yva.astype('int8'), y_score=val_s,                    y_pred=predth.astype('int64'), thresholds=th,                    classes=np.array(SYMPTOM_9))print('저장:', f'{DIR}/m3_val_pred_gpu.npz')print('\n로컬에서 할 일:')print('  1) Drive 에서 m3_val_pred_gpu.npz 를 내려받아 cache/m3_val_pred.npz 로 저장')print('  2) python -m src.viz.plots m3')print('  3) python -m src.viz.inspect_errors m3 --cls 오심 --n 5')